In [2]:
import os
from pathlib import Path

def count_yolo_dataset(dataset_path):
    dataset_path = Path(dataset_path)
    splits = ["train", "valid", "test"]

    counts = {}
    for split in splits:
        # Look for split folders anywhere inside dataset_path
        img_dirs = list(dataset_path.rglob(f"{split}/images"))
        lbl_dirs = list(dataset_path.rglob(f"{split}/labels"))

        img_count = sum(len(list(d.glob("*.jpg"))) + len(list(d.glob("*.png"))) for d in img_dirs)
        lbl_count = sum(len(list(d.glob("*.txt"))) for d in lbl_dirs)

        # Only include split if any images or labels were found
        if img_count > 0 or lbl_count > 0:
            counts[split] = {
                "images": img_count,
                "labels": lbl_count
            }

    return counts

datasets = [
    "my-raccoons.v1i.yolov11",
    "Raccoon.v2-raw.yolov11",
    "Rodent.v2i.yolov11",
    "Rodents.v2i.yolov11",
    "combined_dataset",
    "rebalanced_dataset",
    "augmentation_dataset"
]

for ds in datasets:
    print(f"Dataset: {ds}")
    try:
        counts = count_yolo_dataset(ds)
        if counts:
            for split, stats in counts.items():
                print(f"  {split}: {stats['images']} images, {stats['labels']} labels")
        else:
            print("  No images/labels found in any split.")
    except Exception as e:
        print(f"  Error reading {ds}: {e}")


Dataset: my-raccoons.v1i.yolov11
  train: 459 images, 459 labels
  valid: 29 images, 29 labels
  test: 17 images, 17 labels
Dataset: Raccoon.v2-raw.yolov11
  train: 150 images, 150 labels
  valid: 29 images, 29 labels
  test: 17 images, 17 labels
Dataset: Rodent.v2i.yolov11
  train: 351 images, 351 labels
  valid: 30 images, 30 labels
  test: 2 images, 2 labels
Dataset: Rodents.v2i.yolov11
  train: 633 images, 633 labels
  valid: 36 images, 36 labels
  test: 27 images, 27 labels
Dataset: combined_dataset
  No images/labels found in any split.
Dataset: rebalanced_dataset
  No images/labels found in any split.
Dataset: augmentation_dataset
  No images/labels found in any split.


In [3]:
import os
import shutil
from pathlib import Path
import uuid
import random
import pandas as pd

def remap_yolo_labels(label_path, new_label_id):
    """
    Helper function to overwrite YOLO label IDs with a new one.
    """
    try:
        with open(label_path, "r") as f:
            lines = f.readlines()
        remapped = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) > 0:
                parts[0] = str(new_label_id)  # replace class id
                remapped.append(" ".join(parts))
        with open(label_path, "w") as f:
            f.write("\n".join(remapped))
    except Exception as e:
        print(f"⚠️ Failed to remap {label_path}: {e}")

def create_combined_dataset(datasets, output_dir="combined_dataset", class_map=None):
    """
    Simple function to combine YOLO datasets with unique IDs
    """
    output_path = Path(output_dir)

    # Create directories
    for split in ["train", "valid", "test"]:
        (output_path / split / "images").mkdir(parents=True, exist_ok=True)
        (output_path / split / "labels").mkdir(parents=True, exist_ok=True)

    mapping = []

    for dataset in datasets:
        dataset_path = Path(dataset)
        print(f"📦 Adding dataset: {dataset_path.name}")

        # infer label id based on name
        label_id = None
        for key, value in (class_map or {}).items():
            if key.lower() in dataset_path.name.lower():
                label_id = value
                break

        if label_id is None:
            print(f"⚠️ No class mapping found for {dataset_path.name}, skipping label remap.")
        
        for split in ["train", "valid", "test"]:
            img_dir = next(dataset_path.rglob(f"{split}/images"), None)
            lbl_dir = next(dataset_path.rglob(f"{split}/labels"), None)

            if img_dir and img_dir.exists():
                for img_file in img_dir.glob("*.*"):
                    if img_file.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                        unique_id = str(uuid.uuid4())
                        new_img_name = f"{unique_id}{img_file.suffix}"
                        new_lbl_name = f"{unique_id}.txt"

                        # Copy image
                        shutil.copy2(img_file, output_path / split / "images" / new_img_name)

                        # Copy and remap label
                        if lbl_dir and lbl_dir.exists():
                            lbl_file = lbl_dir / f"{img_file.stem}.txt"
                            if lbl_file.exists():
                                new_lbl_path = output_path / split / "labels" / new_lbl_name
                                shutil.copy2(lbl_file, new_lbl_path)
                                if label_id is not None:
                                    remap_yolo_labels(new_lbl_path, label_id)

                        mapping.append({
                            "dataset": dataset_path.name,
                            "new_id": unique_id,
                            "split": split,
                            "class_id": label_id
                        })

    df = pd.DataFrame(mapping)
    df.to_csv(output_path / "file_mapping.csv", index=False)

    print(f"\n✅ Created combined dataset in: {output_dir}")
    print(f"📊 Total images: {len(mapping)}")
    return output_path

# Usage:
datasets = [
    "my-raccoons.v1i.yolov11",
    "Raccoon.v2-raw.yolov11", 
    "Rodent.v2i.yolov11",
    "Rodents.v2i.yolov11"
]

class_map = {
    "raccoon": 0,
    "rodent": 1
}

combined_path = create_combined_dataset(datasets, class_map=class_map)

📦 Adding dataset: my-raccoons.v1i.yolov11
📦 Adding dataset: Raccoon.v2-raw.yolov11
📦 Adding dataset: Rodent.v2i.yolov11
📦 Adding dataset: Rodents.v2i.yolov11

✅ Created combined dataset in: combined_dataset
📊 Total images: 1780


In [4]:
import cv2
from pathlib import Path
import tqdm
import shutil

def convert_yolo_dataset_to_grayscale(input_dataset="combined_dataset",
                                      output_dataset="combined_dataset_grayscale"):
    """
    Converts all images in a YOLO-style dataset (train/val/test) to grayscale.
    Copies labels unchanged and writes grayscale images to a new output dataset folder.

    Parameters:
    - input_dataset (str): Path to the original dataset.
    - output_dataset (str): Path to save the grayscale-converted dataset.
    """

    input_path = Path(input_dataset)
    output_path = Path(output_dataset)

    if not input_path.exists():
        raise FileNotFoundError(f"❌ Input dataset not found: {input_dataset}")

    # Create output folder structure
    for split in ["train", "valid", "test"]:
        img_in = input_path / split / "images"
        lbl_in = input_path / split / "labels"
        img_out = output_path / split / "images"
        lbl_out = output_path / split / "labels"

        if not img_in.exists():
            print(f"⚠️ Skipping {split}: no images found.")
            continue

        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)

        print(f"\n🎨 Converting {split} images to grayscale...")

        # Convert all images to grayscale (3-channel)
        for img_path in tqdm.tqdm(list(img_in.glob("*.*"))):
            if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
                continue

            img = cv2.imread(str(img_path))
            if img is None:
                continue

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            gray_3ch = cv2.merge([gray, gray, gray])  # keep 3 channels
            cv2.imwrite(str(img_out / img_path.name), gray_3ch)

        # Copy labels unchanged
        if lbl_in.exists():
            for lbl_file in lbl_in.glob("*.txt"):
                shutil.copy(lbl_file, lbl_out / lbl_file.name)

    print(f"\n✅ Grayscale dataset saved to: {output_path.resolve()}")

    return output_path

combined_g_path = convert_yolo_dataset_to_grayscale(
    input_dataset="combined_dataset",
    output_dataset="combined_dataset_grayscale"
)



🎨 Converting train images to grayscale...


100%|██████████| 1593/1593 [00:08<00:00, 178.09it/s]



🎨 Converting valid images to grayscale...


100%|██████████| 124/124 [00:00<00:00, 180.09it/s]



🎨 Converting test images to grayscale...


100%|██████████| 63/63 [00:00<00:00, 191.14it/s]



✅ Grayscale dataset saved to: C:\Users\plebj\Desktop\School\CS5814\combined_dataset_grayscale


In [5]:
def rebalance_train_test_split(dataset_path, output_dir="rebalanced_dataset", train_ratio=0.70, valid_ratio=0.20, test_ratio=0.10):
    """
    Rebalance a YOLO dataset into a new train/test split (default 70/20/10).
    Works on the already combined dataset without duplicating counts.
    """
    dataset_path = Path(dataset_path)
    output_path = Path(output_dir)

    # Collect all images + labels just once
    all_pairs = []
    for split in ["train", "valid", "test"]:
        img_dir = dataset_path / split / "images"
        lbl_dir = dataset_path / split / "labels"
        if not img_dir.exists():
            continue
        for img_file in img_dir.glob("*.*"):
            if img_file.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                lbl_file = lbl_dir / f"{img_file.stem}.txt"
                if lbl_file.exists():
                    all_pairs.append((img_file, lbl_file))

    print(f"Found {len(all_pairs)} unique image-label pairs in total.")

    # Shuffle
    random.shuffle(all_pairs)

    # Split indices
    n = len(all_pairs)
    train_end = int(n * train_ratio)
    valid_end = train_end + int(n * valid_ratio)

    # Train/Test/Valid split
    train_pairs = all_pairs[:train_end]
    valid_pairs = all_pairs[train_end:valid_end]
    test_pairs  = all_pairs[valid_end:]

    # Reset rebalanced folder
    if output_path.exists():
        shutil.rmtree(output_path)

    for split in ["train", "valid", "test"]:
        (output_path / split / "images").mkdir(parents=True, exist_ok=True)
        (output_path / split / "labels").mkdir(parents=True, exist_ok=True)

    # Copy files into new dataset
    def copy_pairs(pairs, split):
        for img_file, lbl_file in pairs:
            shutil.copy2(img_file, output_path / split / "images" / img_file.name)
            shutil.copy2(lbl_file, output_path / split / "labels" / lbl_file.name)

    copy_pairs(train_pairs, "train")
    copy_pairs(valid_pairs, "valid")
    copy_pairs(test_pairs, "test")

    print(f"Created rebalanced dataset at: {output_path}")
    print(f"{len(train_pairs)} train, {len(valid_pairs)} valid, {len(test_pairs)} test")
    return output_path

rebalanced_path = rebalance_train_test_split(combined_g_path)

Found 1780 unique image-label pairs in total.
Created rebalanced dataset at: rebalanced_dataset
1246 train, 356 valid, 178 test


In [6]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import numpy as np
from pathlib import Path
import shutil
import uuid
import pandas as pd

class RandomLightingShift(A.ImageOnlyTransform):
    """
    Simulates lighting domain shifts: day → dusk → night transitions.
    - Adjusts brightness, gamma, and adds subtle tinting.
    - Helps models generalize to different lighting domains.
    """
    def __init__(self, p=0.3):
        super().__init__(p)
        self.modes = ['day', 'dusk', 'night']

    def apply(self, img, mode=None, **params):
        if mode is None:
            mode = np.random.choice(self.modes)
        img = img.astype(np.float32) / 255.0

        if mode == 'day':
            img = np.clip(img * np.random.uniform(1.0, 1.1), 0, 1)

        elif mode == 'dusk':
            # Warm tint + slight darkening
            tint = np.array([1.05, 0.9, 0.8])  # warm sunset tone
            img = np.clip(img * tint * np.random.uniform(0.8, 0.95), 0, 1)

        elif mode == 'night':
            # Darken + bluish tint + gamma compression
            tint = np.array([0.8, 0.85, 1.1])  # cool night tone
            img = np.clip(img * tint * np.random.uniform(0.4, 0.7), 0, 1)
            gamma = np.random.uniform(0.6, 0.9)
            img = np.power(img, gamma)

        img = (img * 255).astype(np.uint8)
        return img


class YOLOAugmentation:
    def __init__(self, augmentation_strength='medium'):
        """Initialize augmentation pipeline based on strength"""
        
        if augmentation_strength == 'light':
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.3),
                A.RandomBrightnessContrast(p=0.2),
                A.HueSaturationValue(p=0.2),
                A.RandomGamma(p=0.2),
            ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

        elif augmentation_strength == 'medium':
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.2),
                A.RandomRotate90(p=0.3),

                # Lighting domain simulation (day/dusk/night)
                RandomLightingShift(p=0.4),

                # Realistic color & exposure changes
                A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=20, val_shift_limit=10, p=0.3),
                A.RandomBrightnessContrast(brightness_limit=(-0.25, 0.25),
                                        contrast_limit=0.2, p=0.4),
                A.RandomGamma(gamma_limit=(80, 120), p=0.3),  # Prevent flash-like overbright

                # Mild blur & motion simulating real capture artifacts
                A.MotionBlur(blur_limit=5, p=0.3),
                A.GaussNoise(std_range=(0.04, 0.15), per_channel=True, p=0.3),

                # Add “light falloff” (vignette-like darkness near edges)
                A.RandomShadow(num_shadows_limit=(1, 2), shadow_roi=(0, 0, 1, 1), p=0.2),


                # Add localized brightness glare occasionally (simulate reflection, not blast)
                A.RandomSunFlare(
                    flare_roi=(0.1, 0.1, 0.9, 0.4),
                    num_flare_circles_range=(2, 4),    # replaces num_flare_circles_lower/upper
                    src_radius=30,
                    p=0.1
                ),


                # Slight compression + occasional grayscale for robustness
                A.ImageCompression(quality_range=(85, 100), p=0.2),
                A.ToGray(p=0.2),
            ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

        


        elif augmentation_strength == 'heavy':
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.3),
                A.RandomRotate90(p=0.5),

                # Lighting domain simulation (day/dusk/night)
                RandomLightingShift(p=0.4),

                # Stronger lighting variation
                A.RandomBrightnessContrast(brightness_limit=(-0.4, 0.4),
                                        contrast_limit=0.3, p=0.6),
                A.RandomGamma(gamma_limit=(60, 140), p=0.5),
                A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=25, val_shift_limit=15, p=0.4),

                # More aggressive environmental noise
                A.MotionBlur(blur_limit=7, p=0.3),
                A.GaussNoise(var_limit=(20, 70), p=0.4),
                A.ISONoise(color_shift=(0.02, 0.06), intensity=(0.2, 0.6), p=0.3),

                # Realistic lighting & shadow patterns
                A.RandomShadow(num_shadows_lower=1, num_shadows_upper=3, shadow_dimension=5, p=0.3),
                A.RandomSunFlare(flare_roi=(0.1, 0.1, 0.9, 0.4),
                                angle_lower=0, angle_upper=1,
                                num_flare_circles_lower=3, num_flare_circles_upper=6,
                                src_radius=40, p=0.15),

                # Minor geometry
                A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=15, p=0.4),

                A.ToGray(p=0.3),
                A.ImageCompression(quality_lower=80, quality_upper=100, p=0.3),
            ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

    
    def read_yolo_label(self, label_path, img_width, img_height):
        """Read YOLO format label file"""
        bboxes = []
        class_labels = []
        
        try:
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    if line.strip():
                        class_id, x_center, y_center, width, height = map(float, line.strip().split())
                        bboxes.append([x_center, y_center, width, height])
                        class_labels.append(class_id)
        except FileNotFoundError:
            pass
            
        return bboxes, class_labels
    
    def write_yolo_label(self, label_path, bboxes, class_labels):
        """Write YOLO format label file"""
        with open(label_path, 'w') as f:
            for bbox, class_id in zip(bboxes, class_labels):
                if all(0 <= coord <= 1 for coord in bbox):  # Ensure normalized coordinates
                    f.write(f"{int(class_id)} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n")
    
    def augment_image_and_labels(self, image_path, label_path):
        """Apply augmentation to image and corresponding labels"""
        # Read image
        image = cv2.imread(str(image_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Read labels
        bboxes, class_labels = self.read_yolo_label(label_path, image.shape[1], image.shape[0])
        
        if not bboxes:  # Skip if no bounding boxes
            return None, None, None
        
        # Apply augmentation
        try:
            augmented = self.transform(image=image, bboxes=bboxes, class_labels=class_labels)
            return augmented['image'], augmented['bboxes'], augmented['class_labels']
        except Exception as e:
            print(f"Augmentation failed for {image_path}: {e}")
            return None, None, None

def augment_yolo_dataset(input_dataset="combined_dataset", 
                        output_dataset="combined_with_augmentation",
                        augment_per_image=3,
                        augmentation_strength='medium',
                        splits_to_augment=['train']):
    """
    Main function to augment YOLO dataset
    
    Args:
        input_dataset: Path to input dataset
        output_dataset: Path to output augmented dataset (will contain originals + augmented)
        augment_per_image: Number of augmented versions per original image
        augmentation_strength: 'light', 'medium', or 'heavy'
        splits_to_augment: Which splits to augment (usually just 'train')
    """
    
    input_path = Path(input_dataset)
    output_path = Path(output_dataset)
    
    # Initialize augmenter
    augmenter = YOLOAugmentation(augmentation_strength)
    
    # Create output directory structure
    splits = ['train', 'valid', 'test']
    for split in splits:
        (output_path / split / 'images').mkdir(parents=True, exist_ok=True)
        (output_path / split / 'labels').mkdir(parents=True, exist_ok=True)
    
    # Track all files
    augmentation_log = []
    stats = {split: {'original': 0, 'augmented': 0} for split in splits}
    
    for split in splits:
        print(f"\n📁 Processing {split} split...")
        
        input_images_dir = input_path / split / 'images'
        input_labels_dir = input_path / split / 'labels'
        
        output_images_dir = output_path / split / 'images'
        output_labels_dir = output_path / split / 'labels'
        
        image_files = list(input_images_dir.glob('*.*'))
        
        for img_path in image_files:
            if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                label_path = input_labels_dir / f"{img_path.stem}.txt"
                
                # ✅ Copy original files to new dataset
                shutil.copy2(img_path, output_images_dir / img_path.name)
                if label_path.exists():
                    shutil.copy2(label_path, output_labels_dir / f"{img_path.stem}.txt")
                    stats[split]['original'] += 1
                
                augmentation_log.append({
                    'original_image': img_path.name,
                    'new_image': img_path.name,
                    'split': split,
                    'type': 'original',
                    'augmentation_id': 'original'
                })
                
                # ✅ Apply augmentation for specified splits
                if split in splits_to_augment and label_path.exists():
                    successful_augmentations = 0
                    
                    for aug_idx in range(augment_per_image):
                        aug_image, aug_bboxes, aug_classes = augmenter.augment_image_and_labels(
                            img_path, label_path
                        )
                        
                        if aug_image is not None and aug_bboxes:
                            # Generate unique ID for augmented file
                            aug_id = str(uuid.uuid4())
                            aug_image_name = f"{aug_id}{img_path.suffix}"
                            aug_label_name = f"{aug_id}.txt"
                            
                            # Save augmented image
                            aug_image_bgr = cv2.cvtColor(aug_image, cv2.COLOR_RGB2BGR)
                            cv2.imwrite(str(output_images_dir / aug_image_name), aug_image_bgr)
                            
                            # Save augmented labels
                            augmenter.write_yolo_label(output_labels_dir / aug_label_name, aug_bboxes, aug_classes)
                            
                            successful_augmentations += 1
                            stats[split]['augmented'] += 1
                            
                            augmentation_log.append({
                                'original_image': img_path.name,
                                'new_image': aug_image_name,
                                'split': split,
                                'type': 'augmented',
                                'augmentation_id': f"aug_{aug_idx+1}"
                            })
                    
                    print(f"  ✅ {img_path.name}: {successful_augmentations}/{augment_per_image} augmentations")
    
    # Save augmentation log
    log_df = pd.DataFrame(augmentation_log)
    log_df.to_csv(output_path / 'augmentation_log.csv', index=False)
    
    # Print summary statistics
    print("\n" + "="*50)
    print("🎉 AUGMENTATION COMPLETE!")
    print("="*50)
    
    total_original = 0
    total_augmented = 0
    
    for split in splits:
        orig = stats[split]['original']
        aug = stats[split]['augmented']
        total_original += orig
        total_augmented += aug
        
        if split in splits_to_augment:
            print(f"📊 {split.upper()}:")
            print(f"   Original images: {orig}")
            print(f"   Augmented images: {aug}")
            print(f"   Total: {orig + aug} images")
            print(f"   Increase: {((orig + aug) / orig - 1) * 100:.1f}%")
        else:
            print(f"📊 {split.upper()}: {orig} images (no augmentation)")
    
    print(f"\n📈 OVERALL STATISTICS:")
    print(f"   Total original images: {total_original}")
    print(f"   Total augmented images: {total_augmented}")
    print(f"   Final dataset size: {total_original + total_augmented} images")
    print(f"   Overall increase: {((total_original + total_augmented) / total_original - 1) * 100:.1f}%")
    
    print(f"\n💾 Output dataset: {output_path}")
    print(f"📋 Augmentation log: {output_path / 'augmentation_log.csv'}")
    
    return output_path


augmented_dataset_path = augment_yolo_dataset( 
    input_dataset="rebalanced_dataset",
    output_dataset="augmentation_dataset",
    augment_per_image=3,  # Create 3 augmented versions per training image
    augmentation_strength='medium',  # 'light', 'medium', or 'heavy'
    splits_to_augment=['train']  # Only augment training split
)


📁 Processing train split...
  ✅ 00022a21-d990-4d5c-ab34-81df8d8a7ef7.jpg: 3/3 augmentations
  ✅ 000e9ad7-ddb1-48a1-9d28-f96a71fcdb45.jpg: 3/3 augmentations
  ✅ 000fadf2-8fa0-479f-a527-1a8320add367.jpg: 3/3 augmentations
  ✅ 001a3c3a-c674-4afe-9019-ff59f39bd0a9.jpg: 3/3 augmentations
  ✅ 001d259e-3433-49a0-89f6-1da6cb0f1305.jpg: 3/3 augmentations
  ✅ 00a0cb01-5fe1-443c-9739-363e1373d097.jpg: 3/3 augmentations
  ✅ 00bf3629-e12a-473a-ab10-0d525b94baf5.jpg: 3/3 augmentations
  ✅ 00f5ca90-93f2-4847-9cdf-5f5dc7f3e6f6.jpg: 3/3 augmentations
  ✅ 0160166f-f557-4b96-95a7-fd636775dd6f.jpg: 3/3 augmentations
  ✅ 01bee47b-73cc-4f7e-a10c-a71afc97fd33.jpg: 3/3 augmentations
  ✅ 01d01a3c-b945-4fd5-b655-4b9e24448de1.jpg: 3/3 augmentations
  ✅ 01d8fef1-6ca7-47c4-880b-ff89acd041b4.jpg: 3/3 augmentations
  ✅ 01f9e7f1-ed82-4958-add0-711505be7af1.jpg: 3/3 augmentations
  ✅ 01fcb485-2bdb-4c2e-b203-bcf3c0d05701.jpg: 3/3 augmentations
  ✅ 0212a55e-8fbb-4aad-b74f-8890069c0045.jpg: 3/3 augmentations
  ✅ 02458d7